# MEGA-RAG: Run Evaluation on Colab GPU

Runs the official PubMedQA benchmark (500 test samples) on Colab's free T4 GPU.

## Two Modes of Running

### Mode A: VS Code Colab Extension (Recommended)
- Install **"Google Colab"** extension in VS Code
- Open this notebook in VS Code → Select Kernel → **Colab** → T4 GPU
- Files stay local, compute runs on Colab GPU
- Set `RUN_MODE = "vscode"` below

### Mode B: Standalone Colab (Upload project)
- Open this notebook directly in colab.research.google.com
- Upload your project as a zip file
- Set `RUN_MODE = "colab_standalone"` below

## Prerequisites
- **API Keys**: Add as Colab Secrets (sidebar key icon):
  - `GEMINI_API_KEY` → https://aistudio.google.com/apikey
  - `GROQ_API_KEY` → https://console.groq.com/keys

**Estimated Time**: ~30 min (quick, 50 samples) / ~2-4 hours (full, 500 samples)

## 1. Configuration & Environment Detection

In [ ]:
# =============================================================================
# CONFIGURATION — Change these settings
# =============================================================================

# Run mode: "vscode" (VS Code Colab extension) or "colab_standalone" (upload zip)
RUN_MODE = "vscode"  # ← Change to "colab_standalone" if running directly in Colab browser

# Evaluation settings
QUICK_TEST = True          # True = 50 samples (~15 min), False = 500 samples (~2-3 hrs)
SAMPLE_SIZE = 50 if QUICK_TEST else 500
EVAL_CONFIGS = "llm_only,oracle_context,lean_workflow"  # Configs to evaluate

# Contextual chunking (uses API calls during indexing — disable for first run)
ENABLE_CONTEXTUAL = False

# =============================================================================
# GROQ RATE LIMIT OPTIMIZATION
# =============================================================================
# Groq free tier: 30 RPM. We stay at 28 RPM to be safe.
# API calls per sample by config:
#   llm_only:        1 call/sample  → 28 samples/min
#   oracle_context:  1 call/sample  → 28 samples/min
#   lean_workflow:   1-2 calls/sample (generation + optional DISC correction)
#
# With self-consistency OFF and single reasoning path:
#   50 samples × 3 configs × ~1.5 calls = ~225 calls → ~9 min
#   500 samples × 3 configs × ~1.5 calls = ~2,250 calls → ~90 min
#
# With self-consistency ON (3 paths):
#   Each yes/no question uses 3x more calls → 3x slower
#   Only enable for final benchmark runs

# =============================================================================
# AUTO-DETECT ENVIRONMENT
# =============================================================================
import os, sys

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')
IS_LOCAL = not IS_COLAB and not IS_KAGGLE

if RUN_MODE == "vscode":
    PROJECT_DIR = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
    if not os.path.exists(os.path.join(PROJECT_DIR, "mega_rag")):
        PROJECT_DIR = os.getcwd()
        if not os.path.exists(os.path.join(PROJECT_DIR, "mega_rag")):
            PROJECT_DIR = "/Users/harsh/major_project/medicalq-a_rag"
    print(f"Mode: VS Code Colab Extension")
    print(f"Project dir: {PROJECT_DIR}")
    print(f"Files: LOCAL  |  Compute: COLAB GPU")

elif RUN_MODE == "colab_standalone":
    PROJECT_DIR = "/content/medicalq-a_rag"
    print(f"Mode: Standalone Colab")
    print(f"Project dir: {PROJECT_DIR}")
    print(f"Files: COLAB  |  Compute: COLAB GPU")

os.chdir(PROJECT_DIR)
print(f"\nWorking directory: {os.getcwd()}")

# GPU check
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)")
else:
    print("WARNING: No GPU detected — embeddings/reranking will be slower")

In [ ]:
# =============================================================================
# PROJECT SETUP (only needed for colab_standalone mode)
# =============================================================================

if RUN_MODE == "colab_standalone":
    # Mount Google Drive for persistent results
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/mega_rag_results'
    os.makedirs(DRIVE_DIR, exist_ok=True)

    # Upload and extract project if not already present
    if not os.path.exists(PROJECT_DIR):
        print("Upload your project zip file.")
        print("Create it locally with:")
        print("  cd /Users/harsh/major_project")
        print("  zip -r medicalq-a_rag.zip medicalq-a_rag/ \\")
        print("    -x 'medicalq-a_rag/venv/*' 'medicalq-a_rag/chroma_*/*' \\")
        print("    -x 'medicalq-a_rag/node_modules/*' 'medicalq-a_rag/.git/*'")
        print()
        from google.colab import files
        uploaded = files.upload()
        !unzip -q medicalq-a_rag.zip -d /content
        os.chdir(PROJECT_DIR)
        print(f"Project extracted to {PROJECT_DIR}")
    else:
        print(f"Project already exists at {PROJECT_DIR}")

elif RUN_MODE == "vscode":
    # In VS Code mode, files are already local — just set results dir
    DRIVE_DIR = os.path.join(PROJECT_DIR, "evaluation_results")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"VS Code mode — results will be saved to: {DRIVE_DIR}")
    print("No upload needed, your local files are accessible.")

In [ ]:
# =============================================================================
# INSTALL DEPENDENCIES
# =============================================================================
# In VS Code mode with local venv active, most deps are already installed.
# On standalone Colab, we need to install everything.

import subprocess, sys

def is_package_installed(name):
    try:
        __import__(name)
        return True
    except ImportError:
        return False

# Check if key packages are already available
if is_package_installed("mega_rag") or is_package_installed("chromadb"):
    print("Dependencies already installed (local venv detected)")
else:
    print("Installing dependencies...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])
    print("All dependencies installed!")

# Add project to Python path (needed if not installed as package)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Verify core imports
from mega_rag.config import LLM_PROVIDER, LLM_FALLBACK_CHAIN
print(f"\nLLM_PROVIDER={LLM_PROVIDER}, chain={LLM_FALLBACK_CHAIN}")

In [ ]:
# =============================================================================
# API KEYS & LLM SETUP (Groq-optimized for rate limits)
# =============================================================================
import os

# Try loading from Colab Secrets first, then .env, then manual
keys_loaded = False

# Method 1: Colab Secrets
try:
    from google.colab import userdata
    groq_key = userdata.get('GROQ_API_KEY')
    if groq_key:
        os.environ['GROQ_API_KEY'] = groq_key
        keys_loaded = True
        print('Groq API key loaded from Colab Secrets')
except Exception:
    pass

# Method 2: .env file
if not keys_loaded:
    env_path = os.path.join(PROJECT_DIR, '.env')
    if os.path.exists(env_path):
        from dotenv import load_dotenv
        load_dotenv(env_path)
        keys_loaded = True
        print(f'API keys loaded from {env_path}')

# Configure: Groq as primary, optimized for 28 RPM
os.environ['LLM_PROVIDER'] = 'groq'
os.environ['GROQ_MODEL'] = 'llama-3.3-70b-versatile'
os.environ['LLM_FALLBACK_CHAIN'] = 'groq'
os.environ['LLM_AUTO_FALLBACK'] = 'false'   # Groq only, no fallback noise
os.environ['ENABLE_CONTEXTUAL_CHUNKING'] = str(ENABLE_CONTEXTUAL).lower()

# Rate-limit-friendly: disable multi-path features during evaluation
# These multiply API calls per sample (3x for self-consistency, 3x for CoT)
os.environ['ENABLE_SELF_CONSISTENCY'] = 'false'    # Saves 2 calls/sample
os.environ['SELF_CONSISTENCY_NUM_PATHS'] = '1'      # Single path
os.environ['ENABLE_CITATION_VERIFICATION'] = 'false' # Saves 1 call/sample

# Verify
has_groq = bool(os.environ.get('GROQ_API_KEY', '').strip().replace('your-groq-api-key-here', ''))
print(f'\nGroq API key: {"SET" if has_groq else "MISSING"}')
print(f'Model: {os.environ["GROQ_MODEL"]}')
print(f'Self-consistency: OFF (saves API calls)')
print(f'Citation verification: OFF (saves API calls)')
print(f'Estimated calls/sample: ~1 (llm_only/oracle) or ~1.5 (lean)')

if not has_groq:
    print('\nAdd GROQ_API_KEY in Colab sidebar → Secrets (key icon)')
    print('Get free key: https://console.groq.com/keys')

In [ ]:
# =============================================================================
# VERIFY DATA FILES
# =============================================================================
import json

print("Data files check:")
for split in ['test', 'dev', 'train_balanced', 'train_oversampled']:
    path = f'pubmedQA/splits/{split}.json'
    if os.path.exists(path):
        with open(path) as f:
            n = len(json.load(f))
        status = "OK" if (split == 'test' and n == 500) or (split == 'dev' and n == 50) or n > 0 else "WARN"
        print(f'  [{status}] {split}.json: {n} samples')
    else:
        print(f'  [FAIL] {split}.json: NOT FOUND!')

print(f"\nEvaluation plan:")
print(f"  Samples: {SAMPLE_SIZE}")
print(f"  Configs: {EVAL_CONFIGS}")
print(f"  Contextual chunking: {ENABLE_CONTEXTUAL}")
print(f"  Run mode: {RUN_MODE}")

## 2. Build PubMedQA Index

Indexes all 1000 PubMedQA contexts (no answers) into ChromaDB + BM25 + Graph.
Takes ~5-10 min without contextual chunking, ~15-30 min with it.

In [ ]:
%%time
# Build the controlled PubMedQA-only index
# Skip if index already exists (saves time on re-runs)

index_dir = "chroma_pubmedqa_only"
if os.path.exists(index_dir) and os.path.exists(f"{index_dir}/bm25_index.pkl"):
    print(f"Index already exists at {index_dir} — skipping build.")
    print("Delete it and re-run to rebuild: !rm -rf chroma_pubmedqa_only")
else:
    !python build_pubmedqa_index.py --force \
        --persist-dir {index_dir} \
        --collection-name pubmedqa_only
    print('\nIndex built successfully!')

## 3. Run Evaluation

Runs the configured evaluation. Change `QUICK_TEST` and `EVAL_CONFIGS` in cell 1 to control scope.

In [ ]:
%%time
# Run evaluation with configured settings

out_file = f"evaluation_results/eval_{'quick' if QUICK_TEST else 'full'}_{SAMPLE_SIZE}.json"
os.makedirs("evaluation_results", exist_ok=True)

!python evaluate_pubmedqa_multiconfig.py \
    --configs {EVAL_CONFIGS} \
    --test-json pubmedQA/splits/test.json \
    --sample-size {SAMPLE_SIZE} \
    --out {out_file} \
    --seed 42

# Display results
import json
try:
    with open(out_file) as f:
        results = json.load(f)

    print('\n' + '='*72)
    print(f'EVALUATION RESULTS ({SAMPLE_SIZE} samples)')
    print('='*72)
    print(f'{"Config":<20} {"Accuracy":>10} {"Macro-F1":>10} {"Latency":>10} {"Coverage":>10}')
    print('-'*72)
    for name, cfg in results.get('configs', {}).items():
        acc = cfg.get('accuracy', 0)
        f1 = cfg.get('macro_f1', 0)
        lat = cfg.get('avg_latency_s', 0)
        cov = cfg.get('coverage', 0)
        print(f'{name:<20} {acc:>9.1%} {f1:>10.3f} {lat:>9.1f}s {cov:>9.1%}')
    print('='*72)
except FileNotFoundError:
    print(f"Results file not found: {out_file}")

## 4. Full Ablation Study (Optional)

Run ALL configs systematically with LaTeX + CSV output for the paper.
Only run this after the quick test passes.

In [ ]:
%%time
# Full ablation study — all configs, 500 samples
# Uncomment to run (takes 2-4 hours)

# !python scripts/run_ablation_study.py \
#     --test-json pubmedQA/splits/test.json \
#     --sample-size 500 \
#     --out-dir evaluation_results

print("Uncomment the command above to run the full ablation study.")

In [ ]:
# Display ablation results (run after ablation study completes)
import json, os

ablation_path = "evaluation_results/ablation_study.json"
if os.path.exists(ablation_path):
    with open(ablation_path) as f:
        study = json.load(f)

    print('='*72)
    print('ABLATION STUDY RESULTS — Official PubMedQA')
    print('='*72)
    print(f'{"Config":<20} {"Accuracy":>10} {"Macro-F1":>10} {"Latency":>10} {"Correct":>10}')
    print('-'*72)
    for r in sorted(study['results'], key=lambda x: x['accuracy'], reverse=True):
        print(f"{r['config']:<20} {r['accuracy']:>9.1%} {r.get('macro_f1',0):>10.3f} "
              f"{r['avg_latency']:>9.1f}s {r['correct']:>6}/{r['total']}")
    print('='*72)

    # Show LaTeX
    latex_path = "evaluation_results/ablation_latex.tex"
    if os.path.exists(latex_path):
        print('\nLaTeX table:')
        with open(latex_path) as f:
            print(f.read())
else:
    print("No ablation results yet. Run the full ablation study first (cell above).")

## 5. Save Results

In **colab_standalone** mode, copies results to Google Drive.
In **vscode** mode, results are already saved locally in `evaluation_results/`.

In [ ]:
import shutil
from datetime import datetime

if RUN_MODE == "colab_standalone":
    # Copy results to Google Drive for persistence
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    dest_dir = f'{DRIVE_DIR}/eval_{timestamp}'
    os.makedirs(dest_dir, exist_ok=True)

    for f in os.listdir('evaluation_results'):
        src = f'evaluation_results/{f}'
        if os.path.isfile(src):
            shutil.copy2(src, dest_dir)
            print(f'  Copied: {f}')

    print(f'\nResults saved to Google Drive: {dest_dir}')

elif RUN_MODE == "vscode":
    print("VS Code mode — results are already in your local evaluation_results/ directory.")
    print("Files:")
    for f in sorted(os.listdir('evaluation_results')):
        if os.path.isfile(f'evaluation_results/{f}'):
            size = os.path.getsize(f'evaluation_results/{f}') / 1024
            print(f'  {f} ({size:.1f} KB)')

## 6. (Optional) Re-run with Contextual Chunking

Rebuild the index WITH contextual chunking, then evaluate again.
This uses ~1000 LLM API calls for indexing — make sure you have quota.

In [ ]:
# %%time
# # Uncomment this entire cell to run with contextual chunking
# # WARNING: Uses ~1000 LLM API calls for context generation
#
# import os
# os.environ['ENABLE_CONTEXTUAL_CHUNKING'] = 'true'
#
# # Rebuild index with contextual descriptions
# !python build_pubmedqa_index.py --force \
#     --persist-dir chroma_pubmedqa_contextual \
#     --collection-name pubmedqa_contextual
#
# # Run evaluation on contextual index
# !python evaluate_pubmedqa_multiconfig.py \
#     --configs oracle_context,hybrid,lean_workflow \
#     --test-json pubmedQA/splits/test.json \
#     --sample-size {SAMPLE_SIZE} \
#     --persist-dir chroma_pubmedqa_contextual \
#     --collection-name pubmedqa_contextual \
#     --out evaluation_results/eval_contextual.json
#
# # Compare results
# import json
# for name, path in [("Without contextual", out_file), ("With contextual", "evaluation_results/eval_contextual.json")]:
#     if os.path.exists(path):
#         with open(path) as f:
#             data = json.load(f)
#         print(f"\n{name}:")
#         for cfg_name, cfg in data.get("configs", {}).items():
#             print(f"  {cfg_name}: acc={cfg.get('accuracy',0):.1%} F1={cfg.get('macro_f1',0):.3f}")

print("Uncomment cell above to test contextual chunking improvement.")